# Dispute Case Workflow Discovery

¿Qué workflow stateful y auditable soportan realmente los datos?

Etapa posterior al EDA/addendum: no rehace profiling y no implementa agentes, modelos, APIs,
políticas, UI ni acciones financieras. Las relaciones complaint–transaction son **candidatas**.
Las cohortes, reglas y arquetipos son HEURISTIC; el diseño de estados es FUTURE DESIGN.
Todos los porcentajes indican denominador; monedas separadas; nulos y ventanas censuradas visibles.


In [1]:
from pathlib import Path
from contextlib import ExitStack
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
assert (ROOT / "AGENTS.md").exists()
assert sys.version_info >= (3, 12)
sys.path.insert(0, str(ROOT))
from src.dispute_workflow_data import ART, sources, cohorts, retrieval, temporal_direction
from src.dispute_workflow_context import (characteristics, behavior, service, transcripts,
    digital, bundles, archetypes)
from src.dispute_workflow_design import future_design, validate, report

def show(name, limit=24):
    # Read only aggregate CSVs; string dtype preserves leading-zero response codes.
    frame = pd.read_csv(ART / name, dtype=str, keep_default_na=False)
    display(frame.head(limit))
    if len(frame) > limit:
        print(f"Displayed {limit}/{len(frame)} aggregate rows. Full table: {name}")

resources = ExitStack()
con, context = resources.enter_context(sources())
display({k:v for k,v in context.items() if k != "protected_artifacts_sha256"})
show("population_cutoff_audit.csv")


Validando 2194 fuentes suplementarias y caché curada de solo lectura


{'execution_timestamp': '2026-10-02T22:51:49.519170-05:00',
 'feature_cutoff_inclusive': '2026-06-17',
 'source_manifest_sha256': 'edaa6b4db2c0053ead935ffc30d0d2928d0d4769bfa8e7b9d880e57043827074',
 'inherited_cache_sha256': 'bd6d87c315103fa6d4e771b52dae2df69a382d7e223378a6464ecb1adecf2b8d',
 'method': 'Exact DuckDB aggregates; no sampling; no raw modification',
 'relationship_class': 'INFERRED RELATIONSHIP / HEURISTIC, never ground truth',
 'temporal_policy': 'Strict event timestamps; no time zone conversion without semantics. Cutoff inclusive by date.',
 'point_in_time_limit': 'Event-time ordered, NOT certified ingestion-time safe: availability/revision timestamps absent.',
 'windows': 'Symmetric +/-24*n hours, inclusive; after-complaint candidates retrospective only.',
 'currency_policy': 'Exact DECIMAL(24,6); compare only same currency; relative difference / abs(claimed_amount), undefined at zero.',
 'input_mode': 'read-only curated DuckDB',
 'transaction_min': '2023-06-17 06:01:30

,dataset,full_rows,cutoff_rows,after_cutoff_rows,invalid_or_missing_timestamp
0,transactions,4425008,4423656,1352,0
1,complaints,67095,67066,29,0
2,call_center_interactions,686296,686121,175,0
3,digital_events,15620994,15618934,2060,0


## 1. Candidate dispute cohorts

A/B/C usan exclusivamente valores observados. B está contenida en A; no sumar cohortes. Las coberturas de resolución y estados son retrospectivas, no evidencia de intake.

In [2]:
cohorts(con, context)
show("candidate_dispute_cohorts.csv")
show("candidate_cohort_definitions.csv")
show("cohort_overlap.csv")
show("candidate_cohort_distributions.csv", 60)
show("retrieval_population.csv")

1. Cohortes y candidatos ±30 días construidos; ninguna relación declarada verdadera


,cohort,records,customers,date_min,date_max,claimed_amount_present,claimed_amount_pct,currency_present,currency_pct,amount_currency_present,resolution_present,resolution_pct
0,A,11571,11137,2023-06-17 15:37:40,2026-06-17 23:34:11,4500,38.8903292714545,4487,38.777979431336966,4302,2721,23.515685766139487
1,B,10477,10121,2023-06-17 15:37:40,2026-06-17 23:34:11,4090,39.03789252648659,4076,38.904266488498614,3907,2459,23.47045910088766
2,C,10373,10022,2023-06-17 13:19:54,2026-06-17 23:10:30,4033,38.87978405475754,4049,39.0340306565121,3822,2319,22.35611684180083


,cohort,sql_predicate,semantic_strength,semantic_weakness,linkage_limitation,potential_role,relationship_class
0,A,"case_type IN ('Complaint','Claim') AND categor...",Explicit transaction category in Claim/Complaint,Includes unspecified subcategory,Customer only; no verified transaction or orig...,Broad sensitivity population,"HEURISTIC cohort, not ground truth"
1,B,"case_type IN ('Complaint','Claim') AND categor...",Explicit recorded Cargo no reconocido,Allegation not adjudication; B is subset of A,Customer only; no verified transaction or orig...,"Primary discovery scope, subject to evidence a...","HEURISTIC cohort, not ground truth"
2,C,"case_type IN ('Complaint','Claim') AND categor...",Explicit recorded Cobro indebido,Fees may not correspond to an individual trans...,Customer only; no verified transaction or orig...,"Separate optional intake lane, not pooled with B","HEURISTIC cohort, not ground truth"


,cohort_left,cohort_right,shared_complaints
0,A,A,11571
1,A,B,10477
2,B,A,10477
3,B,B,10477
4,C,C,10373


,cohort,field,value,records,pct_cohort
0,A,status,Closed,476,4.11373260738052
1,A,status,Escalated,576,4.977962146746176
2,A,status,In Process,4583,39.607639789127994
3,A,status,Open,3441,29.738138449572208
4,A,status,Rejected,106,0.9160833117275948
5,A,status,Resolved,2389,20.64644369544551
6,B,status,Closed,431,4.113773026629761
7,B,status,Escalated,529,5.049155292545576
8,B,status,In Process,4153,39.63920969743247
9,B,status,Open,3102,29.607712131335305


,unique_complaints,complaint_candidate_pairs,unique_candidate_transactions,same_customer_history_transactions
0,21944,35698,35583,311099


## 2. Customer transaction search space

customer_id es el único enlace de búsqueda. Ventanas simétricas de ±24*n horas inclusivas. La censura en extremos queda contada; cero candidatos no es un NULL.

In [3]:
retrieval(con, context)
show("complaint_transaction_window_summary.csv")

2–3,12. Ambigüedad, elegibilidad y reglas monetarias por moneda; sin pesos ni ranking entrenado


,cohort,days,eligible_complaints,zero_candidates,unique_candidate,multiple_candidates,candidate_count_mean,candidate_count_median,candidate_count_p90,candidate_count_p95,candidate_count_max,complete_window_complaints,boundary_censored_complaints
0,A,1,11571,10961,587,23,0.05479215279578256,0.0,0.0,1.0,3,11556,15
1,A,3,11571,9900,1497,174,0.16091954022988506,0.0,1.0,1.0,4,11516,55
2,A,7,11571,8194,2593,784,0.376371964393743,0.0,1.0,2.0,5,11423,148
3,A,14,11571,6042,3339,2190,0.764670296430732,0.0,2.0,3.0,8,11290,281
4,A,30,11571,3549,2958,5064,1.6450609281825253,1.0,4.0,5.0,15,10958,613
5,B,1,10477,9928,531,18,0.054213992555120744,0.0,0.0,1.0,3,10464,13
6,B,3,10477,8980,1345,152,0.15872864369571443,0.0,1.0,1.0,4,10427,50
7,B,7,10477,7440,2337,700,0.37243485730648085,0.0,1.0,2.0,5,10341,136
8,B,14,10477,5481,3018,1978,0.7611911806814928,0.0,2.0,3.0,8,10221,256
9,B,30,10477,3205,2687,4585,1.6411186408322993,1.0,4.0,5.0,15,9921,556


## 3. Claimed amount + currency narrowing

Comparaciones DECIMAL exactas dentro de moneda; tolerancias 1 unidad, 1% y 5% explícitas. Las relativas usan abs(claimed_amount) y no son elegibles en cero. Un candidato único no es un match verdadero.

In [4]:
show("amount_currency_candidate_matching.csv", 45)
show("candidate_rules_common_eligibility.csv", 20)

,cohort,days,currency,rule,cohort_currency_complaints,eligible_complaints,ineligible_complaints,no_candidate,unique_candidate,multiple_candidates,median_candidate_set_size,candidate_rate_pct,ambiguity_pct_eligible,ambiguity_pct_with_candidates
0,A,1,ARS,absolute_1,1114,1061,53,1061,0,0,0.0,0.0,0.0,
1,A,1,ARS,exact,1114,1061,53,1061,0,0,0.0,0.0,0.0,
2,A,1,ARS,relative_1pct,1114,1061,53,1061,0,0,0.0,0.0,0.0,
3,A,1,ARS,relative_5pct,1114,1061,53,1061,0,0,0.0,0.0,0.0,
4,A,1,COP,absolute_1,1121,1080,41,1080,0,0,0.0,0.0,0.0,
5,A,1,COP,exact,1121,1080,41,1080,0,0,0.0,0.0,0.0,
6,A,1,COP,relative_1pct,1121,1080,41,1080,0,0,0.0,0.0,0.0,
7,A,1,COP,relative_5pct,1121,1080,41,1080,0,0,0.0,0.0,0.0,
8,A,1,MXN,absolute_1,1132,1081,51,1081,0,0,0.0,0.0,0.0,
9,A,1,MXN,exact,1132,1081,51,1081,0,0,0.0,0.0,0.0,


Displayed 45/300 aggregate rows. Full table: amount_currency_candidate_matching.csv


,cohort,days,currency,rule,eligible_complaints,no_candidate,unique_candidate,multiple_candidates,median_candidate_set_size
0,A,1,ARS,absolute_1,1061,1061,0,0,0.0
1,A,1,ARS,exact,1061,1061,0,0,0.0
2,A,1,ARS,prior_currency,1061,1049,12,0,0.0
3,A,1,ARS,prior_exact,1061,1061,0,0,0.0
4,A,1,ARS,prior_relative_5pct,1061,1061,0,0,0.0
5,A,1,ARS,prior_temporal,1061,1027,33,1,0.0
6,A,1,ARS,relative_1pct,1061,1061,0,0,0.0
7,A,1,ARS,relative_5pct,1061,1061,0,0,0.0
8,A,1,ARS,same_currency,1061,1044,17,0,0.0
9,A,1,ARS,temporal,1061,1001,57,3,0.0


Displayed 20/600 aggregate rows. Full table: candidate_rules_common_eligibility.csv


## 4. Temporal directionality

Día calendario y timestamp son ejes diferentes. Bins contiguos sin huecos; proximidad no causalidad. Transacciones posteriores al reclamo son retrospectivas e inadmisibles al intake.

In [5]:
temporal_direction(con)
show("temporal_candidate_distribution.csv", 40)

4. Dirección temporal por día calendario y tiempo transcurrido; asociación no causal


,cohort,calendar_direction,elapsed_direction,candidate_pairs,complaints_with_candidates,pct_cohort_candidate_pairs
0,A,calendar_after,after_complaint_up_to_30d,9274,5659,48.720777515103755
1,A,calendar_before,before_0_1d,160,159,0.8405568689256633
2,A,calendar_before,before_14_30d,5160,3808,27.10795902285264
3,A,calendar_before,before_1_3d,608,594,3.1941161019175204
4,A,calendar_before,before_3_7d,1243,1155,6.530076175466246
5,A,calendar_before,before_7_14d,2263,1966,11.88862621486735
6,A,calendar_same_day,after_complaint_up_to_30d,154,150,0.8090359863409509
7,A,calendar_same_day,before_0_1d,173,170,0.9088521145258734
8,B,calendar_after,after_complaint_up_to_30d,8383,5116,48.75537978364546
9,B,calendar_before,before_0_1d,145,144,0.8433174363149936


## 5. Candidate transaction attributes

Comparar poblaciones deduplicadas: todas al cutoff; historia del mismo cliente antes de su último reclamo de interés; candidatos únicos ±30d. La historia agregada de comparación no es feature as-of por reclamo.

In [6]:
characteristics(con)
show("candidate_transaction_characteristics.csv", 40)

5,7. Distribuciones comparables y señal descriptiva sin usar fraud_score


,population,field,value,records,pct_population,valid_recorded_labels,fraud_recorded_count,recorded_fraud_pct
0,all_cutoff_transactions,transaction_type,Adjustment,132076,2.98567519716723,132076,144,0.10902813531603016
1,all_cutoff_transactions,transaction_type,Deposit,609221,13.771889134236478,609221,607,0.09963543607328047
2,all_cutoff_transactions,transaction_type,Payment,738762,16.700258790466528,738762,730,0.09881396173598533
3,all_cutoff_transactions,transaction_type,Purchase,1083082,24.483865834052196,1083082,1086,0.10026941635074722
4,all_cutoff_transactions,transaction_type,Transfer,896129,20.25765565857743,896129,797,0.08893808815471879
5,all_cutoff_transactions,transaction_type,Withdrawal,964386,21.80065538550014,964386,951,0.09861196657769815
6,all_cutoff_transactions,transaction_status,Approved,4069449,91.9928900438913,4069449,3985,0.09792480505346056
7,all_cutoff_transactions,transaction_status,Declined,221164,4.999575012161886,221164,215,0.09721292796295962
8,all_cutoff_transactions,transaction_status,Pending,88312,1.9963577638044188,88312,79,0.0894555666274119
9,all_cutoff_transactions,transaction_status,Reversed,44731,1.011177180142398,44731,36,0.08048109811987213


Displayed 40/147 aggregate rows. Full table: candidate_transaction_characteristics.csv


## 6. Behavioral context feasibility

Solo timestamps estrictamente anteriores; misma moneda para medianas. Se conserva NULL sin historia monetaria o sin merchant. Contadores cero describen ausencia observada. Revisiones/availability desconocidas impiden certificar PIT productivo.

In [7]:
behavior(con)
show("behavioral_feature_feasibility.csv")

6–7. Contexto estrictamente anterior; censura de 7/30/90 días explícita


,feature,candidate_transactions,computable_count,computable_pct,complete_history_window_count,computable_complete_window_count,observed_zero_count,business_interpretation,recommendation,computability,leakage_risk,classification
0,customer_transaction_count_prior_7d,35583,35583,100.0,35438,35438,27151,Activity frequency,recommended,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."
1,customer_transaction_count_prior_30d,35583,35583,100.0,34844,34844,12707,Activity frequency,recommended,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."
2,customer_transaction_count_prior_90d,35583,35583,100.0,32885,32885,3330,Activity frequency,recommended,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."
3,median_amount_prior_30d,35583,21876,61.47879605429559,34844,21578,0,Same-currency historical amount only,recommended,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."
4,median_amount_prior_90d,35583,31621,88.86546946575612,32885,29741,0,Same-currency historical amount only,recommended,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."
5,same_channel_count_prior_90d,35583,35583,100.0,32885,32885,16896,Familiarity with recorded channel,recommended,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."
6,same_transaction_type_count_prior_90d,35583,35583,100.0,32885,32885,16749,Familiarity with recorded type,recommended,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."
7,same_merchant_count_prior_90d,35583,8375,23.53652024843324,32885,7759,7914,Exact observed merchant string; not verified m...,optional,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."
8,prior_declined_count_30d,35583,35583,100.0,34844,34844,33485,Earlier events with current recorded status; s...,optional,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."
9,prior_reversed_count_30d,35583,35583,100.0,34844,34844,35155,Earlier events with current recorded status; s...,optional,Exact event-time range aggregate; count=0 is o...,Strictly prior events only; availability/revis...,"DERIVED FEATURE feasibility, not production/mo..."


## 7. Fraud signal feasibility — without fraud_score

Clases gravemente desbalanceadas. Comparaciones descriptivas por moneda y completitud de historia; sin significancia predictiva, modelo ni threshold. status también requiere revisión de disponibilidad histórica.

In [8]:
show("fraud_signal_without_score.csv", 40)
show("fraud_amount_without_score_by_currency.csv")
show("fraud_prior_behavior_without_score.csv", 30)

,population,field,value,records,pct_population,valid_recorded_labels,fraud_recorded_count,recorded_fraud_pct
0,all_cutoff_transactions,transaction_type,Adjustment,132076,2.98567519716723,132076,144,0.10902813531603016
1,all_cutoff_transactions,transaction_type,Deposit,609221,13.771889134236478,609221,607,0.09963543607328047
2,all_cutoff_transactions,transaction_type,Payment,738762,16.700258790466528,738762,730,0.09881396173598533
3,all_cutoff_transactions,transaction_type,Purchase,1083082,24.483865834052196,1083082,1086,0.10026941635074722
4,all_cutoff_transactions,transaction_type,Transfer,896129,20.25765565857743,896129,797,0.08893808815471879
5,all_cutoff_transactions,transaction_type,Withdrawal,964386,21.80065538550014,964386,951,0.09861196657769815
6,all_cutoff_transactions,transaction_status,Approved,4069449,91.9928900438913,4069449,3985,0.09792480505346056
7,all_cutoff_transactions,transaction_status,Declined,221164,4.999575012161886,221164,215,0.09721292796295962
8,all_cutoff_transactions,transaction_status,Pending,88312,1.9963577638044188,88312,79,0.0894555666274119
9,all_cutoff_transactions,transaction_status,Reversed,44731,1.011177180142398,44731,36,0.08048109811987213


Displayed 40/135 aggregate rows. Full table: fraud_signal_without_score.csv


,population,currency,is_fraud,records,mean,median,p75,p90,p95,p99,minimum,maximum
0,all_cutoff_transactions,ARS,False,791551,586311.2173285992,163283.070000,693012.655000,1791412.210000,2643262.100000,3326320.055000,1750.050000,3499939.110000
1,all_cutoff_transactions,ARS,True,787,584111.879656925,161173.890000,664946.710000,1885254.869999,2607117.400999,3373965.398599,1806.070000,3498771.630000
2,all_cutoff_transactions,COP,False,1192970,6697128.752798411,1867104.450000,7916655.102500,20418267.460000,30218740.352500,38028508.992300,20006.900000,39999828.480000
3,all_cutoff_transactions,COP,True,1114,6581971.061876122,1925248.990000,7806701.457500,19531406.200000,28757217.214999,38998237.968299,21546.590000,39963368.450000
4,all_cutoff_transactions,USD,False,2434820,1675.7354755423398,467.090000,1979.140000,5120.521000,7551.380499,9509.338100,5.000000,9999.980000
5,all_cutoff_transactions,USD,True,2414,1517.1294324772161,444.400000,1817.350000,4558.855000,6694.603499,9298.885899,5.230000,9926.700000
6,unique_candidates_plus_minus_30d,ARS,False,6544,572130.6086506724,162454.365000,675378.877500,1728888.782000,2590234.142000,3314880.563900,1876.430000,3499638.890000
7,unique_candidates_plus_minus_30d,ARS,True,7,375070.8642857143,127085.380000,483398.160000,908385.822000,1093156.190999,1240972.486199,10695.500000,1277926.560000
8,unique_candidates_plus_minus_30d,COP,False,9890,6638071.057573306,1856585.380000,7892641.940000,19936815.868000,30389339.024999,37727834.870500,21410.510000,39998261.190000
9,unique_candidates_plus_minus_30d,COP,True,6,2329809.553333333,1345271.555000,1828512.232500,5340504.855000,7048826.902500,8415484.540500,296426.060000,8757148.950000


,feature,currency,is_fraud,complete_history_window,candidates,observed,mean,median,p75,p90,p95,p99,minimum,maximum
0,customer_transaction_count_prior_7d,ARS,False,False,23,23,0.13043478260869565,0.0,0.0,0.8000000000000007,1.0,1.0,0,1
1,customer_transaction_count_prior_7d,ARS,False,True,6521,6521,0.28707253488728723,0.0,0.0,1.0,1.0,2.0,0,4
2,customer_transaction_count_prior_7d,ARS,True,True,7,7,0.14285714285714285,0.0,0.0,0.40000000000000036,0.6999999999999993,0.9399999999999995,0,1
3,customer_transaction_count_prior_7d,COP,False,False,41,41,0.0975609756097561,0.0,0.0,0.0,1.0,1.0,0,1
4,customer_transaction_count_prior_7d,COP,False,True,9849,9849,0.29637526652452023,0.0,0.0,1.0,1.0,2.0,0,5
5,customer_transaction_count_prior_7d,COP,True,True,6,6,0.16666666666666666,0.0,0.0,0.5,0.75,0.9500000000000002,0,1
6,customer_transaction_count_prior_7d,USD,False,False,81,81,0.1728395061728395,0.0,0.0,1.0,1.0,1.2000000000000028,0,2
7,customer_transaction_count_prior_7d,USD,False,True,19036,19036,0.27054002941794497,0.0,0.0,1.0,1.0,2.0,0,5
8,customer_transaction_count_prior_7d,USD,True,True,19,19,0.2631578947368421,0.0,0.0,1.0,1.0999999999999979,1.8200000000000003,0,2
9,customer_transaction_count_prior_30d,ARS,False,False,121,121,0.628099173553719,0.0,1.0,2.0,2.0,3.0,0,4


Displayed 30/100 aggregate rows. Full table: fraud_prior_behavior_without_score.csv


## 8. Service interaction proximity

Ventanas posteriores disjuntas [0,24h], (24,72h], (72,168h], (168,336h]. Mismo cliente y cercanía temporal no prueban referencia a la transacción. Denominadores de pares y contactos únicos separados.

In [9]:
service(con)
show("service_interaction_proximity.csv")
show("service_proximity_distributions.csv", 30)

8. Contactos posteriores: solo asociación temporal del mismo cliente


,time_window,candidate_transactions,fully_observed_window_transactions,transactions_with_interaction,transaction_coverage_pct,transaction_interaction_pairs,distinct_interactions,escalation_valid_pair_labels,escalated_pairs,escalation_pct_pairs,resolution_valid_pair_labels,resolved_pairs,resolution_pct_pairs
0,0_24h,35583,35557,166,0.4665149088047663,166,163,166,20,12.048192771084338,166,126,75.90361445783132
1,1_3d,35583,35521,313,0.8796335328668184,313,301,313,22,7.0287539936102235,313,234,74.76038338658147
2,3_7d,35583,35453,578,1.6243711884888852,582,540,582,34,5.841924398625429,582,433,74.39862542955326
3,7_14d,35583,35308,993,2.790658460500801,1017,879,1017,103,10.127826941986234,1017,779,76.59783677482793


,time_window,field,value,transaction_interaction_pairs,distinct_interactions,pct_pairs
0,0_24h,reason_category,Comercial,11,11,6.626506024096385
1,0_24h,reason_category,Producto,22,22,13.25301204819277
2,0_24h,reason_category,Queja,26,26,15.662650602409638
3,0_24h,reason_category,Retención,6,6,3.6144578313253013
4,0_24h,reason_category,Transaccional,73,71,43.975903614457835
5,0_24h,reason_category,Técnico,28,27,16.867469879518072
6,0_24h,contact_reason,Comercial,11,11,6.626506024096385
7,0_24h,contact_reason,Producto,22,22,13.25301204819277
8,0_24h,contact_reason,Queja,26,26,15.662650602409638
9,0_24h,contact_reason,Retención,6,6,3.6144578313253013


Displayed 30/91 aggregate rows. Full table: service_proximity_distributions.csv


## 9. Transcript availability

La disponibilidad usa interacciones únicas, no pares repetidos. Metadatos excluidos de la caché se recuperan como flags de presencia desde raw, sin exponer su contenido. []/{} no cuentan como contenido.

In [10]:
transcripts(con)
show("transcript_feasibility.csv")
show("transcript_metadata_feasibility.csv")
show("transcript_selection_comparison.csv", 40)

9. Cobertura de transcripts y metadatos; no lectura/exposición de texto personal


,relevant_unique_interactions,interactions_with_transcript,transcript_available_rate_pct
0,1624,410,25.246305418719214


,field,schema_status,relevant_interactions,relevant_transcripts,nonnull_transcripts,nonempty_transcripts,nonempty_pct_transcripts,nonempty_pct_interactions
0,detected_keywords,present_in_raw_not_in_original_curated_projection,1624,410,392,392,95.60975609756098,24.137931034482758
1,mentioned_entities,present_in_raw_not_in_original_curated_projection,1624,410,358,358,87.3170731707317,22.04433497536946
2,detected_intents,present_in_raw_not_in_original_curated_projection,1624,410,386,386,94.14634146341463,23.76847290640394
3,main_topics,present_in_raw_not_in_original_curated_projection,1624,410,410,410,100.0,25.246305418719214


,field,value,transcript_available,interactions,pct_within_value
0,reason_category,Comercial,False,103,77.44360902255639
1,reason_category,Comercial,True,30,22.55639097744361
2,reason_category,Producto,False,257,73.63896848137536
3,reason_category,Producto,True,92,26.361031518624642
4,reason_category,Queja,False,207,72.88732394366197
5,reason_category,Queja,True,77,27.112676056338028
6,reason_category,Retención,False,49,81.66666666666667
7,reason_category,Retención,True,11,18.333333333333332
8,reason_category,Transaccional,False,419,77.0220588235294
9,reason_category,Transaccional,True,125,22.977941176470587


## 10. Identified digital context

Se excluyen eventos anónimos sin identity stitching. Ventanas before anidadas, no aditivas; after es retrospectiva. product_id y event_value no se interpretan como enlace causal ni importe de transacción.

In [11]:
digital(con, context)
show("digital_context_feasibility.csv")
show("digital_event_value_availability.csv")
show("digital_context_distributions.csv", 25)

10. Contexto digital identificado, ventanas anidadas no aditivas, sin identity stitching


,time_window,candidate_transactions,fully_observed_window_transactions,transactions_with_digital_context,transaction_coverage_pct,transaction_event_pairs,distinct_identified_events,pairs_with_recorded_product_id,product_id_pct_pairs
0,before_30m,35583,35582,11,0.0309136385352556,70,70,3,4.285714285714286
1,before_2h,35583,35578,36,0.10117190793356379,275,275,27,9.818181818181818
2,before_24h,35583,35570,338,0.9498918022651266,2665,2630,240,9.00562851782364
3,after_24h,35583,35562,304,0.8543405558834275,2494,2431,226,9.061748195669606


,unique_nearby_identified_events,product_id_present,event_value_present,limitation
0,4938,449,236,event_value units and meaning not established;...


,time_window,field,value,event_pairs,pct_event_pairs
0,before_30m,event_type,Click,14,20.0
1,before_30m,event_type,Error,2,2.857142857142857
2,before_30m,event_type,FormSubmit,4,5.714285714285714
3,before_30m,event_type,Login,11,15.714285714285714
4,before_30m,event_type,Logout,11,15.714285714285714
5,before_30m,event_type,PageView,27,38.57142857142857
6,before_30m,event_type,Purchase,1,1.4285714285714286
7,before_30m,event_category,Authentication,22,31.428571428571427
8,before_30m,event_category,Navigation,18,25.714285714285715
9,before_30m,event_category,Product,16,22.857142857142858


Displayed 25/102 aggregate rows. Full table: digital_context_distributions.csv


## 11. Derived evidence bundles

Grano complaint/candidate_transaction, sin campo de match verdadero. Agregados independientes evitan multiplicación de eventos. Conteos hasta intake y retrospectivos separados. case_register conserva casos sin candidato. Los Parquet con IDs no se muestran ni se usan para seleccionar clientes demo.

In [12]:
bundles(con)
show("evidence_bundle_summary.csv")

11. Bundles analíticos trazables y registro separado de casos sin candidato


,complaint_candidate_pairs,complaints_with_candidate,unique_candidate_transactions,pairs_not_after_intake,future_pairs_retrospective_only,pairs_with_service_observed_by_intake,pairs_with_identified_digital_observed_by_intake
0,35698,15131,35583,17897,17801,825,318


## 12. Transparent candidate narrowing — no learned ranking

Reglas individuales y combinaciones, sin pesos ni probabilidades. La tabla common_eligibility mantiene la misma población al comparar reglas; antes del reclamo se evalúa también por separado.

In [13]:
show("candidate_rule_comparison.csv", 40)
show("candidate_rules_common_eligibility.csv", 40)

,cohort,days,currency,rule,cohort_currency_complaints,eligible_complaints,ineligible_complaints,no_candidate,unique_candidate,multiple_candidates,median_candidate_set_size,candidate_rate_pct,ambiguity_pct_eligible,ambiguity_pct_with_candidates
0,A,1,ARS,absolute_1,1114,1061,53,1061,0,0,0.0,0.0,0.0,
1,A,1,ARS,exact,1114,1061,53,1061,0,0,0.0,0.0,0.0,
2,A,1,ARS,prior_currency,1114,1114,0,1101,13,0,0.0,1.1669658886894076,0.0,0.0
3,A,1,ARS,prior_exact,1114,1061,53,1061,0,0,0.0,0.0,0.0,
4,A,1,ARS,prior_relative_5pct,1114,1061,53,1061,0,0,0.0,0.0,0.0,
5,A,1,ARS,prior_temporal,1114,1114,0,1078,35,1,0.0,3.2315978456014363,0.08976660682226212,2.7777777777777777
6,A,1,ARS,relative_1pct,1114,1061,53,1061,0,0,0.0,0.0,0.0,
7,A,1,ARS,relative_5pct,1114,1061,53,1061,0,0,0.0,0.0,0.0,
8,A,1,ARS,same_currency,1114,1114,0,1096,18,0,0.0,1.6157989228007181,0.0,0.0
9,A,1,ARS,temporal,1114,1114,0,1052,59,3,0.0,5.565529622980251,0.26929982046678635,4.838709677419355


Displayed 40/750 aggregate rows. Full table: candidate_rule_comparison.csv


,cohort,days,currency,rule,eligible_complaints,no_candidate,unique_candidate,multiple_candidates,median_candidate_set_size
0,A,1,ARS,absolute_1,1061,1061,0,0,0.0
1,A,1,ARS,exact,1061,1061,0,0,0.0
2,A,1,ARS,prior_currency,1061,1049,12,0,0.0
3,A,1,ARS,prior_exact,1061,1061,0,0,0.0
4,A,1,ARS,prior_relative_5pct,1061,1061,0,0,0.0
5,A,1,ARS,prior_temporal,1061,1027,33,1,0.0
6,A,1,ARS,relative_1pct,1061,1061,0,0,0.0
7,A,1,ARS,relative_5pct,1061,1061,0,0,0.0
8,A,1,ARS,same_currency,1061,1044,17,0,0.0
9,A,1,ARS,temporal,1061,1001,57,3,0.0


Displayed 40/600 aggregate rows. Full table: candidate_rules_common_eligibility.csv


## 13. Observed case archetypes

Arquetipos heurísticos solapados. Se exportan únicamente los existentes, con población y manejo sugerido. No se usa el nombre high-confidence para una coincidencia sin ground truth.

In [14]:
archetypes(con)
show("case_archetypes.csv", 30)

13. Arquetipos existentes, poblaciones agregadas; ningún cliente seleccionado para demo


,cohort,cohort_complaints,population_size,pct_cohort,archetype,representative_evidence_pattern,automation,uncertainty,recommended_handling,classification
0,A,11571,3301,28.528217094460288,unique_prior_temporal_candidate,One same-customer transaction in prior 30d; am...,Show candidate and exact evidence,Uniqueness is not match correctness,Customer/authorized-human confirmation; preser...,HEURISTIC overlapping archetypes; no individua...
1,B,10477,3015,28.777321752410042,unique_prior_temporal_candidate,One same-customer transaction in prior 30d; am...,Show candidate and exact evidence,Uniqueness is not match correctness,Customer/authorized-human confirmation; preser...,HEURISTIC overlapping archetypes; no individua...
2,C,10373,2992,28.8441145281018,unique_prior_temporal_candidate,One same-customer transaction in prior 30d; am...,Show candidate and exact evidence,Uniqueness is not match correctness,Customer/authorized-human confirmation; preser...,HEURISTIC overlapping archetypes; no individua...
3,A,11571,2126,18.373520006913836,no_candidate_under_monetary_tolerance,Prior candidates exist but none meet explorato...,Explain narrowing result and preserve original...,Tolerance is unapproved; no proof complaint is...,Clarify amount/currency or human review; never...,HEURISTIC overlapping archetypes; no individua...
4,B,10477,1932,18.440393242340363,no_candidate_under_monetary_tolerance,Prior candidates exist but none meet explorato...,Explain narrowing result and preserve original...,Tolerance is unapproved; no proof complaint is...,Clarify amount/currency or human review; never...,HEURISTIC overlapping archetypes; no individua...
5,C,10373,1878,18.104694880940905,no_candidate_under_monetary_tolerance,Prior candidates exist but none meet explorato...,Explain narrowing result and preserve original...,Tolerance is unapproved; no proof complaint is...,Clarify amount/currency or human review; never...,HEURISTIC overlapping archetypes; no individua...
6,A,11571,2460,21.260046668395127,multiple_prior_candidates,Multiple same-customer prior transactions in 30d,Return bounded candidate list,No true transaction label,Clarify intake or human investigation,HEURISTIC overlapping archetypes; no individua...
7,B,10477,2209,21.084279851102416,multiple_prior_candidates,Multiple same-customer prior transactions in 30d,Return bounded candidate list,No true transaction label,Clarify intake or human investigation,HEURISTIC overlapping archetypes; no individua...
8,C,10373,2104,20.283428130723994,multiple_prior_candidates,Multiple same-customer prior transactions in 30d,Return bounded candidate list,No true transaction label,Clarify intake or human investigation,HEURISTIC overlapping archetypes; no individua...
9,A,11571,5810,50.211736237144585,no_prior_candidate,No same-customer transaction in prior 30d,Record explicit empty retrieval,"Outside horizon, incorrect clues or absent dat...",Clarify; human if unresolved,HEURISTIC overlapping archetypes; no individua...


## 14. Workflow responsibility matrix

FUTURE DESIGN: dueños propuestos y riesgos, no capacidades ya implementadas o evaluadas. ML no es dependencia aprobada; LLM no controla autenticación, ownership, política o dinero.

In [15]:
future_design()
show("workflow_responsibility_matrix.csv")

14–16. Responsabilidades, máquina de estados propuesta y opciones MVP; sin implementación


,workflow_step,data_support,proposed_owner,confidence,reason,main_limitation,classification
0,authenticate customer,"external/system, absent from dataset",DETERMINISTIC,HIGH for required boundary,Must use authenticated principal; customer_id ...,External IAM integration; no dataset-based sub...,"FUTURE DESIGN, not implemented"
1,retrieve transaction,strong structural IDs,DETERMINISTIC,HIGH,Exact ID lookup and explicit missing result,No automatic identification from proximity,"FUTURE DESIGN, not implemented"
2,verify ownership,validated transaction/product/customer consist...,DETERMINISTIC,HIGH within dataset,Compare authenticated principal to returned owner,Authentication and current access policy remai...,"FUTURE DESIGN, not implemented"
3,identify candidate transaction,partial; ambiguity quantified by rule tables,DETERMINISTIC,MEDIUM,Transparent customer/time/currency/amount retr...,No verified complaint-transaction labels,"FUTURE DESIGN, not implemented"
4,confirm transaction selection,candidate evidence only,HUMAN,REQUIRED,Customer confirmation or documented authorized...,Unique candidate is not a true-dispute label,"FUTURE DESIGN, not implemented"
5,extract complaint clues,recorded taxonomy; raw text exists but not ana...,LLM,UNVALIDATED,Future structured extraction with evidence spa...,No LLM evaluation or privacy-approved text pip...,"FUTURE DESIGN, not implemented"
6,compute behavioral context,strict prior-event aggregates with censoring f...,DETERMINISTIC,MEDIUM,Counts and same-currency summaries reproducible,No availability/revision timestamps; not certi...,"FUTURE DESIGN, not implemented"
7,risk assessment,recorded rare label; no demonstrated generaliz...,ML,LOW / NOT APPROVED,Only a future research option after provenance...,Exclude fraud_score; post-event/synthetic-labe...,"FUTURE DESIGN, not implemented"
8,summarize evidence,structured evidence bundle available,LLM,UNVALIDATED,Potential grounded narrative and explicit unkn...,Outputs require source references and evaluati...,"FUTURE DESIGN, not implemented"
9,apply policy,policy unavailable,EXTERNAL_POLICY_SERVICE,REQUIRED,Versioned authoritative policy; deterministic ...,Missing policy must block financial eligibilit...,"FUTURE DESIGN, not implemented"


## 15. Proposed case state machine

Solo especificación CSV/JSON. La falta de política/autoridad obliga revisión humana. HANDOFF_RECORDED es distinto de RESOLVED/CLOSED; cada transición define evidencia, guardas y ruta de fallo.

In [16]:
show("proposed_case_state_machine.csv", 30)

,source_state,target_state,trigger,required_evidence,deterministic_guard,AI_allowed,human_required,failure_state,classification
0,CASE_CREATED,AUTH_REQUIRED,Intake received,request_id; untrusted claimed identity,Generate case/request ID and persist untrusted...,False,False,AUTH_REQUIRED,FUTURE DESIGN ONLY
1,AUTH_REQUIRED,INTAKE_INCOMPLETE,External identity verification succeeds,signed current principal; authorization scope,"Verify issuer, expiry, scope and subject via e...",False,False,AUTH_FAILED,FUTURE DESIGN ONLY
2,AUTH_REQUIRED,AUTH_FAILED,Identity verification fails,auth failure record,Deny account reads; audit failure without leak...,False,False,AUTH_FAILED,FUTURE DESIGN ONLY
3,AUTH_FAILED,AUTH_REQUIRED,Authorized retry,new valid auth attempt,Rate limits and external retry policy; no bypa...,False,True,AUTH_FAILED,FUTURE DESIGN ONLY
4,INTAKE_INCOMPLETE,TRANSACTION_SEARCH,Minimum clues supplied,authenticated customer plus transaction_id OR ...,Validate types/units; preserve missing amount/...,True,True,INTAKE_INCOMPLETE,FUTURE DESIGN ONLY
5,TRANSACTION_SEARCH,NO_CANDIDATE,Empty eligible retrieval,versioned query and count=0,Only same authorized customer and nonfuture ad...,False,False,INSUFFICIENT_EVIDENCE,FUTURE DESIGN ONLY
6,TRANSACTION_SEARCH,MULTIPLE_CANDIDATES,More than one eligible candidate,candidate IDs and deterministic rule trace,Retain alternatives; no weighted score promote...,False,False,INSUFFICIENT_EVIDENCE,FUTURE DESIGN ONLY
7,TRANSACTION_SEARCH,TRANSACTION_SELECTED,One candidate explicitly confirmed,candidate evidence + recorded customer/human c...,Count=1 is insufficient alone; require attribu...,False,True,HUMAN_REVIEW,FUTURE DESIGN ONLY
8,MULTIPLE_CANDIDATES,TRANSACTION_SELECTED,Explicit disambiguation,recorded selection by customer/authorized human,Selected ID must belong to admissible retrieve...,False,True,HUMAN_REVIEW,FUTURE DESIGN ONLY
9,MULTIPLE_CANDIDATES,HUMAN_REVIEW,Ambiguity remains,alternatives; missing clues; retrieval trace,Do not infer a true match,True,True,HUMAN_REVIEW,FUTURE DESIGN ONLY


## 16. MVP recommendation and validation

Comparar inquiry, investigation, intake+handoff y adjudicación autónoma. El informe distingue hechos, hipótesis y diseño futuro y responde las diez preguntas de cierre.

In [17]:
show("mvp_scope_assessment.csv")
validate(con)
findings = report(context)
resources.close()
print((ART / "integrity.json").read_text())
display(Markdown(findings))

,option,scope,data_feasibility,engineering_complexity,evaluation_feasibility,risk,hackathon_value,main_blocker,recommendation,classification
0,A,Transaction inquiry only,HIGH,LOW,HIGH for lookup/ownership invariants,LOW with external auth,MEDIUM,Does not demonstrate stateful dispute handling,Fallback,FUTURE DESIGN
1,B,Transaction investigation + structured evidenc...,HIGH with optional gaps,MEDIUM,HIGH for evidence fidelity; LLM fidelity needs...,MEDIUM,HIGH,No verified dispute verdict or policy; optiona...,Investigation subworkflow,FUTURE DESIGN
2,C,Dispute intake + candidate retrieval + human h...,MEDIUM with explicit ambiguity,MEDIUM,HIGH for guards/provenance; unknown linkage pr...,CONTROLLED with mandatory human boundary,HIGH,No reliable complaint-transaction labels; need...,"RECOMMENDED, with B when a transaction is expl...",FUTURE DESIGN
3,D,Autonomous dispute adjudication,UNSUPPORTED,HIGH,UNSUPPORTED for correctness,UNACCEPTABLE on present evidence,Misleading without policy/ground truth,"No verified outcome linkage, policy, auth or f...",NO-GO,FUTURE DESIGN


775 controles de consistencia correctos


Verificando fuentes originales y resultados anteriores sin cambios


{
  "all_raw_size_mtime_unchanged": 7671,
  "supplemental_raw_sha256_unchanged": 2194,
  "inherited_cache_sha256_unchanged": true,
  "prior_artifacts_sha256_unchanged": 53
}



# Dispute Case Workflow Findings

## 1. Executive conclusion

**GO WITH CONSTRAINTS** for a stateful dispute-intake / candidate-retrieval / evidence / human-handoff
system. **NO-GO for autonomous adjudication or financial action.** The dataset supplies grounded
transaction records but does not supply complaint-to-transaction truth, authentication, policy,
decision authority or an operational case transition log.

Executed: 2026-10-02T22:51:49.519170-05:00. Inherited cutoff: 2026-06-17 inclusive.
Transactions/complaints/service/digital after this cutoff are audited and excluded from this stage.
No general EDA or profiling is rerun. See population_cutoff_audit.csv for exact exclusions.
All computations are exhaustive over the stated populations; no sampling or demo-customer selection.

Classification convention throughout:
OBSERVED FACT = supplied field; DERIVED FEATURE = documented aggregate;
HEURISTIC = retrieval rule/cohort/archetype; INFERRED RELATIONSHIP = same-customer temporal association;
HYPOTHESIS = possible later interpretation; FUTURE DESIGN = unimplemented architecture.
No unique candidate becomes a verified dispute link.

## 2. Dispute cohort recommendation

| cohort | records | customers | date_min | date_max | claimed_amount_present | claimed_amount_pct | currency_present | currency_pct | amount_currency_present | resolution_present | resolution_pct |
|---|---|---|---|---|---|---|---|---|---|---|---|
| A | 11571 | 11137 | 2023-06-17 15:37:40 | 2026-06-17 23:34:11 | 4500 | 38.8903292714545 | 4487 | 38.777979431336966 | 4302 | 2721 | 23.515685766139487 |
| B | 10477 | 10121 | 2023-06-17 15:37:40 | 2026-06-17 23:34:11 | 4090 | 39.03789252648659 | 4076 | 38.904266488498614 | 3907 | 2459 | 23.47045910088766 |
| C | 10373 | 10022 | 2023-06-17 13:19:54 | 2026-06-17 23:10:30 | 4033 | 38.87978405475754 | 4049 | 39.0340306565121 | 3822 | 2319 | 22.35611684180083 |

| cohort | sql_predicate | semantic_strength | semantic_weakness | linkage_limitation | potential_role | relationship_class |
|---|---|---|---|---|---|---|
| A | case_type IN ('Complaint','Claim') AND category='Transactions' | Explicit transaction category in Claim/Complaint | Includes unspecified subcategory | Customer only; no verified transaction or origin-interaction link | Broad sensitivity population | HEURISTIC cohort, not ground truth |
| B | case_type IN ('Complaint','Claim') AND category='Transactions' AND subcategory='Cargo no reconocido' | Explicit recorded Cargo no reconocido | Allegation not adjudication; B is subset of A | Customer only; no verified transaction or origin-interaction link | Primary discovery scope, subject to evidence and human review | HEURISTIC cohort, not ground truth |
| C | case_type IN ('Complaint','Claim') AND category='Fees' AND subcategory='Cobro indebido' | Explicit recorded Cobro indebido | Fees may not correspond to an individual transaction | Customer only; no verified transaction or origin-interaction link | Separate optional intake lane, not pooled with B | HEURISTIC cohort, not ground truth |

Recommend **B** as the primary discovery/development intake scope because its recorded allegation
is specific, not because of its size. Use **A** as a broader sensitivity population (B is a subset,
so counts must not be added). Keep **C** as a separate optional fees lane: the fee may not map to
a transaction amount. None is a gold-truth dispute population.
Status, priority, reception channel and repeat-complainer distributions are in
candidate_cohort_distributions.csv. These are snapshot descriptions, not proven creation-time values.
Resolution coverage is retrospective observability only; no resolution information enters bundles.

## 3. Transaction candidate retrieval

Trusted linkage is complaint.customer_id only. Neither affected_product_id nor origin_interaction_id
is used. The former has inherited ownership mismatches; the latter is entirely null.
Search uses exact timestamps in symmetric ±24*n-hour windows; boundaries are inclusive.
Historical same-customer baseline means transactions before each customer's last in-scope complaint,
deduplicated by transaction ID. It is a descriptive comparison, not a per-complaint as-of feature.

| cohort | days | eligible_complaints | zero_candidates | unique_candidate | multiple_candidates | candidate_count_mean | candidate_count_median | candidate_count_p90 | candidate_count_p95 | candidate_count_max | complete_window_complaints | boundary_censored_complaints |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| A | 1 | 11571 | 10961 | 587 | 23 | 0.05479215279578256 | 0.0 | 0.0 | 1.0 | 3 | 11556 | 15 |
| A | 3 | 11571 | 9900 | 1497 | 174 | 0.16091954022988506 | 0.0 | 1.0 | 1.0 | 4 | 11516 | 55 |
| A | 7 | 11571 | 8194 | 2593 | 784 | 0.376371964393743 | 0.0 | 1.0 | 2.0 | 5 | 11423 | 148 |
| A | 14 | 11571 | 6042 | 3339 | 2190 | 0.764670296430732 | 0.0 | 2.0 | 3.0 | 8 | 11290 | 281 |
| A | 30 | 11571 | 3549 | 2958 | 5064 | 1.6450609281825253 | 1.0 | 4.0 | 5.0 | 15 | 10958 | 613 |
| B | 1 | 10477 | 9928 | 531 | 18 | 0.054213992555120744 | 0.0 | 0.0 | 1.0 | 3 | 10464 | 13 |
| B | 3 | 10477 | 8980 | 1345 | 152 | 0.15872864369571443 | 0.0 | 1.0 | 1.0 | 4 | 10427 | 50 |
| B | 7 | 10477 | 7440 | 2337 | 700 | 0.37243485730648085 | 0.0 | 1.0 | 2.0 | 5 | 10341 | 136 |
| B | 14 | 10477 | 5481 | 3018 | 1978 | 0.7611911806814928 | 0.0 | 2.0 | 3.0 | 8 | 10221 | 256 |
| B | 30 | 10477 | 3205 | 2687 | 4585 | 1.6411186408322993 | 1.0 | 4.0 | 5.0 | 15 | 9921 | 556 |
| C | 1 | 10373 | 9829 | 520 | 24 | 0.054757543622867055 | 0.0 | 0.0 | 1.0 | 2 | 10354 | 19 |
| C | 3 | 10373 | 8842 | 1374 | 157 | 0.1639834184903114 | 0.0 | 1.0 | 1.0 | 4 | 10323 | 50 |
| C | 7 | 10373 | 7251 | 2437 | 685 | 0.3816639352164273 | 0.0 | 1.0 | 2.0 | 5 | 10246 | 127 |
| C | 14 | 10373 | 5418 | 2966 | 1989 | 0.758893280632411 | 0.0 | 2.0 | 3.0 | 8 | 10126 | 247 |
| C | 30 | 10373 | 3264 | 2559 | 4550 | 1.6063819531475947 | 1.0 | 4.0 | 5.0 | 12 | 9802 | 571 |

For B, ±1 day gives zero/unique/multiple =
**9928/531/18**
out of **10477** complaints; ±30 days gives
**3205/2687/4585**
out of the same stated cohort. Full-window and boundary-censored denominators are explicit.
These are candidate-search outcomes, **not recall, accuracy or proof that a transaction was disputed**.
Restricting B to the **prior 30 days** gives no/unique/multiple =
**5253/3015/2209**
out of **10477** complaints. This removes future candidates, not uncertainty.

### Amount and currency

Exact comparison uses DECIMAL(24,6), validated against source precision; no binary-float equality.
Currency must be equal. Absolute tolerance is at most 1 unit **of that currency**.
Relative tolerances are abs(transaction amount - claim amount)/abs(claim amount) <=1% or <=5%;
zero claim amounts are ineligible for relative rules. No FX conversion, no silent preferred tolerance.
Missing clues remain in the ineligible count; they are not counted as failed matches.
All monetary rule tables are stratified by currency.

| currency | rule | eligible_complaints | ineligible_complaints | no_candidate | unique_candidate | multiple_candidates | candidate_rate_pct | ambiguity_pct_eligible |
|---|---|---|---|---|---|---|---|---|
| ARS | absolute_1 | 956 | 50 | 956 | 0 | 0 | 0.0 | 0.0 |
| ARS | exact | 956 | 50 | 956 | 0 | 0 | 0.0 | 0.0 |
| ARS | relative_1pct | 956 | 50 | 956 | 0 | 0 | 0.0 | 0.0 |
| ARS | relative_5pct | 956 | 50 | 956 | 0 | 0 | 0.0 | 0.0 |
| COP | absolute_1 | 977 | 35 | 977 | 0 | 0 | 0.0 | 0.0 |
| COP | exact | 977 | 35 | 977 | 0 | 0 | 0.0 | 0.0 |
| COP | relative_1pct | 977 | 35 | 977 | 0 | 0 | 0.0 | 0.0 |
| COP | relative_5pct | 977 | 35 | 977 | 0 | 0 | 0.0 | 0.0 |
| MXN | absolute_1 | 989 | 45 | 989 | 0 | 0 | 0.0 | 0.0 |
| MXN | exact | 989 | 45 | 989 | 0 | 0 | 0.0 | 0.0 |
| MXN | relative_1pct | 989 | 45 | 989 | 0 | 0 | 0.0 | 0.0 |
| MXN | relative_5pct | 989 | 45 | 989 | 0 | 0 | 0.0 | 0.0 |
| USD | absolute_1 | 985 | 39 | 984 | 1 | 0 | 0.10152284263959391 | 0.0 |
| USD | exact | 985 | 39 | 985 | 0 | 0 | 0.0 | 0.0 |
| USD | relative_1pct | 985 | 39 | 982 | 3 | 0 | 0.30456852791878175 | 0.0 |
| USD | relative_5pct | 985 | 39 | 963 | 21 | 1 | 2.233502538071066 | 0.10152284263959391 |
| NULL | absolute_1 | 0 | 6401 | 0 | 0 | 0 | NULL | NULL |
| NULL | exact | 0 | 6401 | 0 | 0 | 0 | NULL | NULL |
| NULL | relative_1pct | 0 | 6401 | 0 | 0 | 0 | NULL | NULL |
| NULL | relative_5pct | 0 | 6401 | 0 | 0 | 0 | NULL | NULL |

B has 0 unique exact candidates and 21 unique <=5% candidates within ±30d,
among 3907 complaints eligible by both amount and currency.
These counts are across distinct currency strata, not pooled monetary distributions.
candidate_rules_common_eligibility.csv holds populations constant across rule comparisons.
candidate_rule_comparison.csv additionally compares before-or-at-complaint rules.
No arbitrary weighted rank or threshold is selected. Status is presented as evidence, not filtered
without a policy basis.

| cohort | currency | complaints | observed_claim_amounts | claim_minimum | claim_median | claim_maximum | transaction_records_same_currency | transaction_minimum | transaction_median | transaction_maximum | support |
|---|---|---|---|---|---|---|---|---|---|---|---|
| B | ARS | 1006 | 956 | 62.530000 | 2502.505000 | 4985.840000 | 792338 | 1750.050000 | 163279.490000 | 3499939.110000 | overlapping_ranges_not_proof_of_linkage |
| B | COP | 1012 | 977 | 55.800000 | 2397.960000 | 4998.130000 | 1194084 | 20006.900000 | 1867144.000000 | 39999828.480000 | disjoint_observed_amount_ranges |
| B | MXN | 1034 | 989 | 55.060000 | 2678.790000 | 4999.900000 | 0 | NULL | NULL | NULL | no_transaction_currency |
| B | USD | 1024 | 985 | 76.190000 | 2552.700000 | 4987.340000 | 2437234 | 5.000000 | 467.070000 | 9999.980000 | overlapping_ranges_not_proof_of_linkage |
| B | NULL | 6401 | 183 | 82.700000 | 2731.380000 | 4961.640000 | 0 | NULL | NULL | NULL | missing_claim_currency |

**Observed currency/amount incompatibility limits narrowing:** MXN has no transactions in this
extract; COP claimed amounts and transaction amounts have disjoint observed ranges. ARS/USD
range overlap is not linkage evidence. These source discrepancies are preserved, not rescaled
or repaired. The almost empty exact-result sets are a limitation, not successful disambiguation.

### Direction and admissibility

temporal_candidate_distribution.csv separates calendar-day direction from elapsed-time bins:
[0,24h], (24,72h], (72,168h], (168,336h], (336,720h] before; exact equality separately;
after-complaint candidates are retrospective sensitivity only. Adjacent bins have no gaps.
An operational search at intake must reject future candidates even if their amount matches.
The data has no linked triggering event, so it cannot validate a causal or optimal retrieval horizon.
The union population has 17897 pairs before/at intake versus
17801 after, out of 35698 pairs:
roughly symmetric proximity does not establish the triggering transaction.

## 4. Evidence available per case

Pair-grain artifact: 35698 candidate pairs,
15131 complaints with a candidate and
35583 distinct candidate transactions.
17801 pairs are after intake and are **not intake-admissible**.
case_register.parquet separately retains zero-candidate complaints; case IDs are derived analytical IDs.

- **Core observed evidence:** transaction ID, customer/product ownership backbone, timestamp,
  amount+currency, type, channel and literal recorded status.
- **Optional/sparse:** merchant, response code, coordinates, historical context, nearby service and
  identified digital activity. Missing optional evidence is not itself suspicious.
- **Recorded-only, quarantined semantics:** is_fraud and fraud_score availability. They are not
  approved risk predictors, probability, legal fraud verdict or action authorization.
- **Unsafe:** complaint/product ownership, complaint/origin contact, true complaint/transaction link;
  anonymous digital identity; assigning a service contact to a transaction from proximity.

| time_window | candidate_transactions | fully_observed_window_transactions | transactions_with_interaction | transaction_coverage_pct | transaction_interaction_pairs | distinct_interactions | escalation_valid_pair_labels | escalated_pairs | escalation_pct_pairs | resolution_valid_pair_labels | resolved_pairs | resolution_pct_pairs |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 0_24h | 35583 | 35557 | 166 | 0.4665149088047663 | 166 | 163 | 166 | 20 | 12.048192771084338 | 166 | 126 | 75.90361445783132 |
| 1_3d | 35583 | 35521 | 313 | 0.8796335328668184 | 313 | 301 | 313 | 22 | 7.0287539936102235 | 313 | 234 | 74.76038338658147 |
| 3_7d | 35583 | 35453 | 578 | 1.6243711884888852 | 582 | 540 | 582 | 34 | 5.841924398625429 | 582 | 433 | 74.39862542955326 |
| 7_14d | 35583 | 35308 | 993 | 2.790658460500801 | 1017 | 879 | 1017 | 103 | 10.127826941986234 | 1017 | 779 | 76.59783677482793 |

Service windows are disjoint [0,24h], (24,72h], (72,168h], (168,336h].
Rates of resolution/escalation use valid labels on transaction-interaction pairs (same interaction
may appear for multiple transactions). Distinct interaction counts are separately reported.
Coverage denominators include all candidate transactions; some boundary windows are partially
observed, so absence of an event in those windows is not proof of a complete negative history.
These outcomes are current recorded flags, not proven available at contact start.

| relevant_unique_interactions | interactions_with_transcript | transcript_available_rate_pct |
|---|---|---|
| 1624 | 410 | 25.246305418719214 |

| field | schema_status | relevant_interactions | relevant_transcripts | nonnull_transcripts | nonempty_transcripts | nonempty_pct_transcripts | nonempty_pct_interactions |
|---|---|---|---|---|---|---|---|
| detected_keywords | present_in_raw_not_in_original_curated_projection | 1624 | 410 | 392 | 392 | 95.60975609756098 | 24.137931034482758 |
| mentioned_entities | present_in_raw_not_in_original_curated_projection | 1624 | 410 | 358 | 358 | 87.3170731707317 | 22.04433497536946 |
| detected_intents | present_in_raw_not_in_original_curated_projection | 1624 | 410 | 386 | 386 | 94.14634146341463 | 23.76847290640394 |
| main_topics | present_in_raw_not_in_original_curated_projection | 1624 | 410 | 410 | 410 | 100.0 | 25.246305418719214 |

Metadata fields exist in raw, not in the original curated projection. Only nonnull/nonempty
indicators were recovered; no raw text or entity values are exposed. Empty lists/objects are not
counted as populated metadata. Existing metadata may itself be generated output, not verified NLP truth.
transcript_selection_comparison.csv compares available/unavailable groups by channel/type/reason;
partial coverage and temporal/customer selection prevent assuming representativeness.

| time_window | candidate_transactions | fully_observed_window_transactions | transactions_with_digital_context | transaction_coverage_pct | transaction_event_pairs | distinct_identified_events | pairs_with_recorded_product_id | product_id_pct_pairs |
|---|---|---|---|---|---|---|---|---|
| before_30m | 35583 | 35582 | 11 | 0.0309136385352556 | 70 | 70 | 3 | 4.285714285714286 |
| before_2h | 35583 | 35578 | 36 | 0.10117190793356379 | 275 | 275 | 27 | 9.818181818181818 |
| before_24h | 35583 | 35570 | 338 | 0.9498918022651266 | 2665 | 2630 | 240 | 9.00562851782364 |
| after_24h | 35583 | 35562 | 304 | 0.8543405558834275 | 2494 | 2431 | 226 | 9.061748195669606 |

Digital before windows are nested and must not be summed; after excludes neither zero timestamp nor
future elapsed time but is explicitly retrospective. Only identified events are used. Product ID
coverage is conditional evidence, not a transaction link. event_value units are undocumented.
**Exclude digital context from the core MVP**, retaining it as optional research context.

Bundles aggregate each domain independently, avoiding service×digital event multiplication.
Intake-time event counts require event_time <= complaint creation; retrospective counts are named
separately. Transcript existence in the extract does not establish transcript availability at intake.
Recorded status/fraud availability times are unknown and flagged explicitly.

## 5. Point-in-time safe behavioral context

| feature | candidate_transactions | computable_count | computable_pct | complete_history_window_count | computable_complete_window_count | recommendation |
|---|---|---|---|---|---|---|
| customer_transaction_count_prior_7d | 35583 | 35583 | 100.0 | 35438 | 35438 | recommended |
| customer_transaction_count_prior_30d | 35583 | 35583 | 100.0 | 34844 | 34844 | recommended |
| customer_transaction_count_prior_90d | 35583 | 35583 | 100.0 | 32885 | 32885 | recommended |
| median_amount_prior_30d | 35583 | 21876 | 61.47879605429559 | 34844 | 21578 | recommended |
| median_amount_prior_90d | 35583 | 31621 | 88.86546946575612 | 32885 | 29741 | recommended |
| same_channel_count_prior_90d | 35583 | 35583 | 100.0 | 32885 | 32885 | recommended |
| same_transaction_type_count_prior_90d | 35583 | 35583 | 100.0 | 32885 | 32885 | recommended |
| same_merchant_count_prior_90d | 35583 | 8375 | 23.53652024843324 | 32885 | 7759 | optional |
| prior_declined_count_30d | 35583 | 35583 | 100.0 | 34844 | 34844 | optional |
| prior_reversed_count_30d | 35583 | 35583 | 100.0 | 34844 | 34844 | optional |

Historical events must satisfy history.timestamp < candidate.timestamp, including exclusion of
same-timestamp events. Lookbacks are 7/30/90 elapsed days. Amount medians compare **only the candidate's
currency**. Merchant counts require an observed merchant; missing merchant yields NULL, not zero.
Counts of zero are exact observed absence within the extract, not imputation.
Early history windows are left-censored and must not be interpreted as full lifetime history.

These features are **event-time safe by construction, not certified availability-time safe**:
process_date has ambiguous semantics and no ingestion/revision timestamp is supplied.
Past transaction status may be revised later; prior_declined/reversed counts remain optional
descriptive evidence until status history is available. Production/model use requires event availability
and revisions to be versioned. No later complaint resolution fields are used as contemporaneous inputs.

## 6. ML feasibility

Recorded positive class: **4315/4423656 (0.097544%)** within this stage's cutoff.
Extreme imbalance must be preserved in evaluation. No classifier, predictive threshold or learned
ranker was trained.

| dimension | minimum_rate_pct | maximum_rate_pct |
|---|---|---|
| transaction_type | 0.08893808815471879 | 0.10902813531603016 |
| channel | 0.09478446387658765 | 0.10795630411970315 |
| transaction_category | 0.09337211670361846 | 0.1068573540963911 |
| transaction_status | 0.08048109811987213 | 0.09792480505346056 |
| merchant_category | 0.09558224706060207 | 0.10789269051321929 |
| transaction_country | 0.09285389590744708 | 0.1234445980643887 |

These group-rate ranges are descriptive, not significance tests; denominators are in
fraud_signal_without_score.csv. Monetary distributions by recorded label remain separated by
currency in fraud_amount_without_score_by_currency.csv. Prior-feature comparisons also show
class counts and history-window completeness in fraud_prior_behavior_without_score.csv.
Candidate-transaction selection is not a representative training population.

**A learned component is not yet justified as an MVP dependency.** Descriptive associations do not
establish incremental/generalizable signal. is_fraud is only a possible future label after provenance,
label-availability and synthetic-construction review. fraud_score is excluded from this feasibility
analysis and from proposed predictors; current transaction status also requires availability review.
Observed categorical rates are narrow and monetary distributions overlap substantially; no clear
non-score separation is demonstrated here. Only 32 recorded positives occur among
35583 unique candidates, leaving little support for interpreting
conditional prior-feature differences. This is descriptive evidence of limited support, not a
statistical proof that no learned signal exists.
The baseline should first be deterministic lookup, transparent retrieval sets, explicit abstention
and handoff. Any later ML baseline must be compared against class prevalence/no-skill behavior on
customer/time-separated held-out data with adequate positive support and no post-event fields.
No linkage labels exist to train/evaluate true transaction matching now.

## 7. AI/LLM feasibility

Potential LLM value is limited to future extraction of user-provided clues, grounded summarization,
explicit unknowns and a draft human handoff. Structured output must be schema-validated and
traceable to evidence IDs; extracted amounts/currencies/IDs require customer confirmation.
Transcripts are partial optional context and require separate privacy/quality evaluation.
No LLM or NLP model ran in this task.

An LLM must never be sole authority for authentication, authorization, ownership, policy eligibility,
financial action or closure. It must not invent code meanings, select a true transaction from an
ambiguous set, infer policy or turn a fraud flag into a verdict. These are future design boundaries,
not demonstrated model capabilities.

## 8. Human escalation boundary

Escalate unresolved no-candidate and multiple-candidate cases, ownership/evidence conflicts,
insufficient required evidence, missing policy, any high-risk financial action and all unsupported
adjudication. Missing optional merchant/geo alone does not mandate a fraud conclusion.
Auth failure blocks data access, rather than granting a human/LLM a bypass.
Even a unique exact candidate requires attributed customer/authorized-human confirmation.

| cohort | archetype | population_size | cohort_complaints | pct_cohort | representative_evidence_pattern | recommended_handling |
|---|---|---|---|---|---|---|
| A | unique_prior_temporal_candidate | 3301 | 11571 | 28.528217094460288 | One same-customer transaction in prior 30d; amount may be absent or inconsistent with heuristic | Customer/authorized-human confirmation; preserve uncertainty |
| B | unique_prior_temporal_candidate | 3015 | 10477 | 28.777321752410042 | One same-customer transaction in prior 30d; amount may be absent or inconsistent with heuristic | Customer/authorized-human confirmation; preserve uncertainty |
| C | unique_prior_temporal_candidate | 2992 | 10373 | 28.8441145281018 | One same-customer transaction in prior 30d; amount may be absent or inconsistent with heuristic | Customer/authorized-human confirmation; preserve uncertainty |
| A | no_candidate_under_monetary_tolerance | 2126 | 11571 | 18.373520006913836 | Prior candidates exist but none meet exploratory same-currency <=5% rule | Clarify amount/currency or human review; never automatic rejection |
| B | no_candidate_under_monetary_tolerance | 1932 | 10477 | 18.440393242340363 | Prior candidates exist but none meet exploratory same-currency <=5% rule | Clarify amount/currency or human review; never automatic rejection |
| C | no_candidate_under_monetary_tolerance | 1878 | 10373 | 18.104694880940905 | Prior candidates exist but none meet exploratory same-currency <=5% rule | Clarify amount/currency or human review; never automatic rejection |
| A | multiple_prior_candidates | 2460 | 11571 | 21.260046668395127 | Multiple same-customer prior transactions in 30d | Clarify intake or human investigation |
| B | multiple_prior_candidates | 2209 | 10477 | 21.084279851102416 | Multiple same-customer prior transactions in 30d | Clarify intake or human investigation |
| C | multiple_prior_candidates | 2104 | 10373 | 20.283428130723994 | Multiple same-customer prior transactions in 30d | Clarify intake or human investigation |
| A | no_prior_candidate | 5810 | 11571 | 50.211736237144585 | No same-customer transaction in prior 30d | Clarify; human if unresolved |
| B | no_prior_candidate | 5253 | 10477 | 50.138398396487545 | No same-customer transaction in prior 30d | Clarify; human if unresolved |
| C | no_prior_candidate | 5277 | 10373 | 50.872457341174204 | No same-customer transaction in prior 30d | Clarify; human if unresolved |
| A | claimed_amount_missing | 7071 | 11571 | 61.1096707285455 | Missing claim amount | Intake incomplete path |
| B | claimed_amount_missing | 6387 | 10477 | 60.96210747351341 | Missing claim amount | Intake incomplete path |
| C | claimed_amount_missing | 6340 | 10373 | 61.12021594524246 | Missing claim amount | Intake incomplete path |
| A | sparse_optional_evidence | 1276 | 11571 | 11.027568922305765 | At least one prior candidate without merchant/score/nearby pre-intake service | Human review if required evidence absent |
| B | sparse_optional_evidence | 1149 | 10477 | 10.966879832012982 | At least one prior candidate without merchant/score/nearby pre-intake service | Human review if required evidence absent |
| C | sparse_optional_evidence | 1073 | 10373 | 10.34416273016485 | At least one prior candidate without merchant/score/nearby pre-intake service | Human review if required evidence absent |
| A | recorded_fraud_flag_present | 9 | 11571 | 0.077780658542909 | Recorded True on a prior candidate | Human risk review, no automatic verdict |
| B | recorded_fraud_flag_present | 8 | 10477 | 0.0763577359931278 | Recorded True on a prior candidate | Human risk review, no automatic verdict |
| C | recorded_fraud_flag_present | 10 | 10373 | 0.09640412609659693 | Recorded True on a prior candidate | Human risk review, no automatic verdict |
| A | service_after_transaction_before_intake | 340 | 11571 | 2.938380433843229 | Same-customer contact temporally after candidate and by intake | Label temporal association in handoff |
| B | service_after_transaction_before_intake | 311 | 10477 | 2.9684069867328433 | Same-customer contact temporally after candidate and by intake | Label temporal association in handoff |
| C | service_after_transaction_before_intake | 294 | 10373 | 2.8342813072399498 | Same-customer contact temporally after candidate and by intake | Label temporal association in handoff |

Archetypes overlap and are reported only when observed. No individual customers were selected.
“Unique candidate” is deliberately not called “high-confidence true match.”

## 9. Proposed case state machine

FUTURE DESIGN ONLY. Main path:

CASE_CREATED → AUTH_REQUIRED → INTAKE_INCOMPLETE → TRANSACTION_SEARCH
→ [NO_CANDIDATE / MULTIPLE_CANDIDATES / explicitly confirmed TRANSACTION_SELECTED]
→ OWNERSHIP_VERIFIED → EVIDENCE_COLLECTION → ASSESSMENT_READY
→ POLICY_REVIEW_REQUIRED → HUMAN_REVIEW → HANDOFF_RECORDED.

Auth failure denies access. Missing core evidence goes through INSUFFICIENT_EVIDENCE.
RESOLVED/CLOSED require external authorized outcome, authoritative policy and audited deterministic
guards; the present dataset cannot supply that authority. HANDOFF_RECORDED is the MVP observable
outcome, **not a fabricated dispute resolution**.

| source_state | target_state | trigger | required_evidence | deterministic_guard | AI_allowed | human_required | failure_state |
|---|---|---|---|---|---|---|---|
| CASE_CREATED | AUTH_REQUIRED | Intake received | request_id; untrusted claimed identity | Generate case/request ID and persist untrusted intake; no account evidence exposed | False | False | AUTH_REQUIRED |
| AUTH_REQUIRED | INTAKE_INCOMPLETE | External identity verification succeeds | signed current principal; authorization scope | Verify issuer, expiry, scope and subject via external auth; never trust LLM identity | False | False | AUTH_FAILED |
| AUTH_REQUIRED | AUTH_FAILED | Identity verification fails | auth failure record | Deny account reads; audit failure without leaking customer data | False | False | AUTH_FAILED |
| AUTH_FAILED | AUTH_REQUIRED | Authorized retry | new valid auth attempt | Rate limits and external retry policy; no bypass through human narrative | False | True | AUTH_FAILED |
| INTAKE_INCOMPLETE | TRANSACTION_SEARCH | Minimum clues supplied | authenticated customer plus transaction_id OR explicit search date/window | Validate types/units; preserve missing amount/currency; confirm extracted clues with user | True | True | INTAKE_INCOMPLETE |
| TRANSACTION_SEARCH | NO_CANDIDATE | Empty eligible retrieval | versioned query and count=0 | Only same authorized customer and nonfuture admissible transactions; expose empty result | False | False | INSUFFICIENT_EVIDENCE |
| TRANSACTION_SEARCH | MULTIPLE_CANDIDATES | More than one eligible candidate | candidate IDs and deterministic rule trace | Retain alternatives; no weighted score promoted to truth | False | False | INSUFFICIENT_EVIDENCE |
| TRANSACTION_SEARCH | TRANSACTION_SELECTED | One candidate explicitly confirmed | candidate evidence + recorded customer/human confirmation | Count=1 is insufficient alone; require attributed selection and authorization | False | True | HUMAN_REVIEW |
| MULTIPLE_CANDIDATES | TRANSACTION_SELECTED | Explicit disambiguation | recorded selection by customer/authorized human | Selected ID must belong to admissible retrieved/explicit-ID result and current owner | False | True | HUMAN_REVIEW |
| MULTIPLE_CANDIDATES | HUMAN_REVIEW | Ambiguity remains | alternatives; missing clues; retrieval trace | Do not infer a true match | True | True | HUMAN_REVIEW |
| NO_CANDIDATE | INTAKE_INCOMPLETE | Customer can clarify clues | new date/amount/currency/ID, with provenance | Never manufacture amount or expand to other customers | True | True | HUMAN_REVIEW |
| NO_CANDIDATE | HUMAN_REVIEW | No further safe clarification | empty-result evidence and limits | No automated denial or reimbursement from missing candidates | True | True | HUMAN_REVIEW |
| TRANSACTION_SELECTED | OWNERSHIP_VERIFIED | Ownership check succeeds | authenticated principal; transaction/product/customer records | Exact subject equality plus validated structural relationship and current access policy | False | False | HUMAN_REVIEW |
| TRANSACTION_SELECTED | HUMAN_REVIEW | Conflicting ownership/evidence | access-safe conflict record | Block selection and financial action; do not repair complaint affected_product_id | False | True | HUMAN_REVIEW |
| OWNERSHIP_VERIFIED | EVIDENCE_COLLECTION | Authorized evidence reads | transaction ID; sources; as-of time | Read allowlisted fields; mark snapshot timing and inferred service/digital links | False | False | INSUFFICIENT_EVIDENCE |
| EVIDENCE_COLLECTION | INSUFFICIENT_EVIDENCE | Required evidence missing | explicit missingness + source errors | Distinguish optional sparsity from required ownership/core evidence absence | False | False | HUMAN_REVIEW |
| EVIDENCE_COLLECTION | ASSESSMENT_READY | Required bundle complete | core facts, optional evidence flags, provenance | Schema/ownership/as-of checks; no post-intake outcomes used as intake facts | False | False | INSUFFICIENT_EVIDENCE |
| INSUFFICIENT_EVIDENCE | HUMAN_REVIEW | Evidence cannot safely be completed | unknowns and conflicts | No fabricated evidence or default fraud verdict | True | True | HUMAN_REVIEW |
| ASSESSMENT_READY | POLICY_REVIEW_REQUIRED | Summary prepared | grounded summary + structured bundle | Validate references and factual consistency; LLM may draft but cannot authorize eligibility | True | False | HUMAN_REVIEW |
| POLICY_REVIEW_REQUIRED | HUMAN_REVIEW | Policy missing, ambiguous or financial decision needed | policy version OR explicit missing-policy flag | Missing policy blocks autonomous decision; all current dataset cases require escalation | False | True | HUMAN_REVIEW |
| HUMAN_REVIEW | HANDOFF_RECORDED | Authorized reviewer/queue accepts handoff | handoff_id; recipient; evidence snapshot; unresolved questions | Persist acceptance and audit receipt; this is NOT a resolved dispute | True | True | HUMAN_REVIEW |
| HANDOFF_RECORDED | HUMAN_REVIEW | Authorized reviewer resumes | assigned actor; preserved evidence; current policy | Idempotent versioned transition with current authorization | False | True | HUMAN_REVIEW |
| HUMAN_REVIEW | RESOLVED | External authorized outcome supplied | signed human outcome; authoritative policy; financial-action approval if applicable | NO dataset-only authority; require external approvals; unsupported until integrated | False | True | HUMAN_REVIEW |
| RESOLVED | CLOSED | Authorized completion recorded | outcome record; approval trail; notification receipt; no pending action | Versioned deterministic closure checks; no LLM-only closure or inferred resolution | False | True | HUMAN_REVIEW |

The JSON specification also defines future audit fields: actor, transition/state version, evidence
references/hashes, event/recorded time, rule/policy/model versions, guard results, human approval and
idempotency key. Dataset complaints.status does not constitute such a transition log.

## 10. Recommended MVP

| option | scope | data_feasibility | engineering_complexity | evaluation_feasibility | risk | hackathon_value | main_blocker | recommendation | classification |
|---|---|---|---|---|---|---|---|---|---|
| A | Transaction inquiry only | HIGH | LOW | HIGH for lookup/ownership invariants | LOW with external auth | MEDIUM | Does not demonstrate stateful dispute handling | Fallback | FUTURE DESIGN |
| B | Transaction investigation + structured evidence summary | HIGH with optional gaps | MEDIUM | HIGH for evidence fidelity; LLM fidelity needs held-out review | MEDIUM | HIGH | No verified dispute verdict or policy; optional context weak | Investigation subworkflow | FUTURE DESIGN |
| C | Dispute intake + candidate retrieval + human handoff | MEDIUM with explicit ambiguity | MEDIUM | HIGH for guards/provenance; unknown linkage prevents true-match accuracy | CONTROLLED with mandatory human boundary | HIGH | No reliable complaint-transaction labels; needs future auth/policy/queue integration | RECOMMENDED, with B when a transaction is explicitly confirmed | FUTURE DESIGN |
| D | Autonomous dispute adjudication | UNSUPPORTED | HIGH | UNSUPPORTED for correctness | UNACCEPTABLE on present evidence | Misleading without policy/ground truth | No verified outcome linkage, policy, auth or financial-action authority | NO-GO | FUTURE DESIGN |

Recommend **Option C**, incorporating Option B only once a transaction has been explicitly selected:

- **Entry point:** authenticated request with transaction ID or customer-confirmed transaction clues;
  B is the primary recorded complaint analogue; A sensitivity and C fees remain separate.
- **Stages:** persisted intake → explicit missing clues → deterministic candidate search →
  confirmation/ownership checks → independently aggregated evidence → optional grounded narrative →
  policy-required human review → auditable handoff receipt.
- **Future tools:** exact transaction lookup, bounded customer/time/currency retrieval, ownership
  check, prior-event context queries, evidence retrieval, case-state/audit store, external IAM,
  authoritative policy interface and human queue. None is implemented here as an agent/API.
- **Deterministic gates:** authentication/authorization, ownership, explicit selection, no future
  evidence, currency/units, schema/provenance validation, version/idempotency checks, missing policy
  blocks action, human approval required for adjudication and closure.
- **Possible ML:** optional later research only; no required fraud/ranking component in the MVP.
- **LLM responsibility:** optional clue extraction and evidence-grounded summary with unknowns;
  never state/financial authority.
- **Human handoff:** unresolved candidate ambiguity, no candidate, conflict, missing required
  evidence/policy or a dispute decision. The dataset cannot safely bypass this boundary.
- **Final observable outcome:** accepted, versioned handoff with evidence IDs, candidate alternatives,
  unknowns and guard results; not automatic reimbursement or claimed correct dispute resolution.

Future evaluation should hold out requests and assess evidence fidelity, ownership/access denial,
temporal/currency invariants, correct handling of empty/ambiguous candidate sets, transition guards,
audit completeness and handoff acceptance. True-match accuracy, retrieval recall and adjudication
correctness need independent verified annotations/policy not supplied here. Human-reviewed or explicitly
labeled synthetic scenarios may test system behavior later, but must not be represented as observed
complaint-to-transaction truth. No demo customers, agents, UI, policy engine or models were built.

### Reproducibility and artifacts

Execute notebooks/03_dispute_case_workflow_discovery.ipynb from top to bottom, or run
python scripts/run_dispute_workflow.py from the repository's Python 3.12+ environment.
All analysis outputs are in [artifacts/dispute_case_workflow](../artifacts/dispute_case_workflow/README.md).
analysis_context.json records cutoff, source signatures and prior artifacts; integrity.json verifies
source/cache/prior-result preservation; validation_checks.json records count and temporal invariants.
No raw data or previous EDA conclusions were changed.
